In [2]:
import sqlite3
import pandas as pd 

import numpy as np
from scipy import stats

con = sqlite3.connect(r'C:\Users\Scalefusion admin\OneDrive\Documents\OJT- SEM 3\Airtel_telecom_Analysis\data\raw\airtel.db')

def q(sql):
    return pd.read_sql(sql, con)

In [3]:
def pval(z, tail='two'):                       # p from a z value
    if tail == 'greater':                      # H1: bigger
        return stats.norm.sf(z)                # area to the right
    if tail == 'less':                         # H1: smaller
        return stats.norm.cdf(z)               # area to the left
    return 2 * stats.norm.sf(abs(z))           # both sides
 
def f_test(x, y):                              # two variances
    f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
    d1, d2 = len(x) - 1, len(y) - 1            # degrees of freedom
    lo = stats.f.cdf(f, d1, d2)                # area to the left
    p = 2 * min(lo, stats.f.sf(f, d1, d2))     # two-tailed
    print(f'F = {f:.4f}  p = {p:.4g}')         # the result
 
def z_mean2(x, y, tail='two'):                 # two means, large
    se = np.sqrt(np.var(x, ddof=1) / len(x)    # standard error,
                 + np.var(y, ddof=1) / len(y)) # each group its own
    z = (np.mean(x) - np.mean(y)) / se         # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

def z_prop2(k1, n1, k2, n2, tail='two'):       # two percentages
    pool = (k1 + k2) / (n1 + n2)               # both groups as one
    se = np.sqrt(pool * (1 - pool) * (1/n1 + 1/n2))  # spread if H0
    z = (k1/n1 - k2/n2) / se                   # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [4]:
# Claim :the two groups used the same amount of data before the offer 
# H₀ :both groups have the same true average daily data before May
# H₁ :the two averages differ
# Tail :two-tailed — a gap either way would break the split
pre=q("""
SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb)  AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY u.customer_id;""")

pre

,g,customer_id,m
0,B,1,729.705882
1,A,2,256.590909
2,A,3,1914.133333
3,B,4,218.880000
4,B,5,980.926667
...,...,...,...
21988,A,21996,6681.156863
21989,A,21997,1460.978571
21990,B,21998,69.021429
21991,A,21999,1153.977778


In [5]:
a=pre[pre.g=='A'].m
b=pre[pre.g=='B'].m

# first we will do f_test that is check the variance of both A and B

f_test(a, b)
# p-value<0.05 hence we will do z_test
z_mean2(a,b)

F = 1.1108  p = 3.673e-08
z = 0.5191  p = 0.6037


In [6]:

cf=q("""
   SELECT * FROM customer_features;""")

A,B=cf[cf.ab_group=='A'],cf[cf.ab_group=='B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)



stated_arpu
tenure_days
complaint_count


In [7]:
cf=q("""
   SELECT * FROM customer_features;""")

A,B=cf[cf.ab_group=='A'],cf[cf.ab_group=='B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)
    f_test(B[col],A[col])

stated_arpu
F = 0.9374  p = 0.0006932
tenure_days
F = 0.9762  p = 0.2061
complaint_count
F = 0.9765  p = 0.2128


In [8]:
rev = q("""
SELECT
    c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r
    ON r.customer_id = c.customer_id
    AND r.recharge_date >= '2025-01-01'
    AND r.recharge_date < '2025-05-01'
GROUP BY c.customer_id;
""")
rev

,g,s
0,B,897
1,A,198
2,A,3408
3,B,0
4,B,1438
...,...,...
21995,A,6891
21996,A,719
21997,B,836
21998,A,798


In [9]:
A = rev[rev.g=='A'].s
B = rev[rev.g=='B'].s

f_test(B,A)

# p-value <0.05 Hence signigicant
# we can do Z_test
z_mean2(B,A)
# p-value>0.05 hence not significant
# FAILWS TO REJECT h0

F = 0.9513  p = 0.008866
z = -1.3490  p = 0.1773


In [10]:
q("""SELECT ab_group, COUNT(*) AS n,
    SUM(device_type = 'Smartphone') AS smart,
    SUM(connection_type = 'Prepaid') AS prepaid
FROM customer_features GROUP BY ab_group;

""")

,ab_group,n,smart,prepaid
0,A,10996,9417,8575
1,B,11004,9554,8574


In [11]:
z_prop2(9554,11004,9417,10996)

# 
z_prop2(8574,11004,8575,10996)

z = 2.5457  p = 0.01091
z = -0.1177  p = 0.9063


In [12]:
# Claim : the offer raised daily data usage
# H₀ :both groups have the same true average daily data in May and June
# H₁ :group B's average is higher
# Tail :one-tailed — the company said so in advance

off=q("""SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date >= '2025-05-01'
GROUP BY u.customer_id;""")
off

,g,customer_id,m
0,A,3,1758.756250
1,B,5,1699.688889
2,A,6,339.763333
3,A,7,5363.175000
4,B,8,1376.940000
...,...,...,...
18443,B,21993,1936.235714
18444,A,21996,6483.050000
18445,A,21997,1262.429032
18446,B,21998,86.217391


In [13]:
A = off[off.g=='A'].m
B = off[off.g=='B'].m

f_test(A,B)
# p-value < 0.05 
z_mean2(A,B)

#hence

F = 0.6197  p = 8.402e-116
z = -19.4496  p = 2.936e-84


In [14]:
# Claim : the offer reduced churn
# H₀ :both groups have the same true churn rate
# H₁ :group B's churn rate is lower
# Tail : one-tailed — the company said so in advance

q("""
SELECT ab_group, COUNT(*) AS n,
      SUM(is_churned)AS all_churn,
      SUM(churn_date >= '2025-05-01')
         AS may_on
FROM customer_features GROUP BY ab_group;

""")

,ab_group,n,all_churn,may_on
0,A,10996,2587,905
1,B,11004,2639,914


In [15]:
# all_churn
z_prop2(2639,11004,2587,10996)

# from may churn

z_prop2(914,11004,905,10996)

# p value > 0.05 not significant
# hence failed to reject H0

z = 0.7937  p = 0.4274
z = 0.2041  p = 0.8382


In [16]:
rev=q("""
        SELECT c.ab_group AS g,
        COALESCE(SUM(r.amount_inr), 0)AS s
FROM customer_features c
LEFT JOIN recharges r ON r.customer_id = c.customer_id
     AND r.recharge_date >= '2025-05-01'
GROUP BY c.customer_id;

""")
rev

,g,s
0,B,0
1,A,0
2,A,1665
3,B,0
4,B,0
...,...,...
21995,A,3587
21996,A,0
21997,B,209
21998,A,0


In [17]:
a = rev[rev.g=='A'].s
b = rev[rev.g=='B'].s
a,b

(1           0
 2        1665
 5         396
 6        3576
 13       3529
          ... 
 21989       0
 21995    3587
 21996       0
 21998       0
 21999       0
 Name: s, Length: 10996, dtype: int64,
 0          0
 3          0
 4          0
 7          0
 8        349
         ... 
 21991    992
 21992    349
 21993      0
 21994      0
 21997    209
 Name: s, Length: 11004, dtype: int64)

In [18]:
f_test(b,a)

# p-value > 0.05 hence
# we will go for t_test

F = 0.9697  p = 0.1068


In [19]:
stats.ttest_ind(b,a,alternative='greater')

TtestResult(statistic=np.float64(-0.5926646653701372), pvalue=np.float64(0.7232941573761509), df=np.float64(21998.0))

In [20]:
# Claim : prepaid and postpaid subscribers churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ
q("""SELECT connection_type AS grp,
            count(*) AS n, SUM(is_churned)AS k
    FROM customer_features
    GROUP  BY connection_type;
""")




,grp,n,k
0,Postpaid,4851,785
1,Prepaid,17149,4441


In [21]:
z_prop2(4441,17149,785,4851)
# p-value < 0.05 so its significant
# HENCE REJECT h0

z = 14.0364  p = 9.333e-45


In [22]:
# Claim :subscribers with an open ticket churn at a different rate from 
# those who never complained
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
SELECT CASE WHEN complaint_count = 0 THEN 'never'
            WHEN unresolved_count = 0 THEN 'all closed'
            ELSE 'open' END AS grp,
        COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features GROUP BY grp;
""")

,grp,n,k
0,all closed,8689,2064
1,never,6631,1328
2,open,6680,1834


In [23]:
z_prop2(1834,6680,1328,6631)

z = 10.0684  p = 7.623e-24


In [24]:
# Claim :year-0 and year-3 subscribers churn at different rates
# H₀ :they share one true churn rate
# H₁ :they do not

q("""
SELECT tenure_years as grp,
        COUNT(*) AS n, SUM(is_churned)AS k
FROM customer_features GROUP BY tenure_years;
""")

,grp,n,k
0,0,4610,1328
1,1,6961,1725
2,2,7055,1541
3,3,3374,632


In [25]:
z_prop2(1328,4610,632,3374)
# p-value < 0.05 so result is significant 
# HENCE REJECT h0

z = 10.3330  p = 4.995e-25


In [26]:
# Claim :Himachal Pradesh and Punjab churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
 SELECT circle_name AS grp,
        COUNT(*) AS n, SUM(is_churned)AS k
    FROM customer_features GROUP BY circle_name;
  """)

,grp,n,k
0,Andhra Pradesh,1412,408
1,Assam,1301,301
2,Bihar,887,195
3,Delhi,735,119
4,Gujarat,1210,298
5,Haryana,1041,265
6,Himachal Pradesh,1247,417
7,Jammu & Kashmir,933,232
8,Karnataka,1361,242
9,Kerala,1141,219


In [27]:
z_prop2(408,1412,77,558)
# p-value < 0.05 so result is significant 
# HENCE REJECT h0

z = 7.0080  p = 2.418e-12


In [28]:
# Claim :smartphone and feature-phone users churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
SELECT device_type AS grp,
        COUNT(*) AS n, SUM(is_churned) AS k
    FROM customer_features
    GROUP BY device_type;
""")

,grp,n,k
0,Feature Phone,3029,719
1,Smartphone,18971,4507


In [29]:
z_prop2(4507,18971,719,3029)
# NOT significant
# FAILED TO REJECT

z = 0.0241  p = 0.9807


In [30]:
# Claim :gender and device type each change the data a subscriber uses
# H₀ :both groups have the same true average daily data
# H₁ :the two averages differ

gen=q("""
SELECT c.gender,c.device_type,
        u.data_mb AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id;
""")
dev=q("""SELECT c.gender,c.device_type,
        AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
GROUP BY c.customer_id;

""")
dev

,gender,device_type,m
0,Male,Smartphone,729.705882
1,Male,Smartphone,256.590909
2,Female,Smartphone,1868.932727
3,Male,Feature Phone,218.880000
4,Female,Smartphone,1146.794872
...,...,...,...
21991,Male,Smartphone,6610.941772
21992,Male,Smartphone,1400.037624
21993,Female,Feature Phone,75.106154
21994,Female,Smartphone,1153.977778


In [31]:
male=gen[gen.gender=='Male'].m
female=gen[gen.gender=='Female'].m

f_test(female,male)
#p value <0.05
#we can go for z test
z_mean2(female,male)
# p value <0.05 so significant

F = 0.9315  p = 6.262e-177
z = -7.2519  p = 4.109e-13


In [32]:
fp=dev[dev.device_type=='Feature Phone'].m
sp=dev[dev.device_type=='Smartphone'].m

f_test(fp,sp)

z_mean2(fp,sp)

F = 0.0268  p = 0
z = -119.2400  p = 0


In [33]:
# C1 · How many payments of each type did we take, and from how many subscribers?
# S T E P S
# 1. Read recharges, where one row is one payment.
# 2. Group by txn_type.
# 3. Count rows and distinct subscribers in each group.

q("""
   SELECT
       txn_type,
       COUNT(DISTINCT customer_id) AS subs,
        COUNT(*) AS payments
   FROM recharges
   GROUP BY txn_type;""")



,txn_type,subs,payments
0,Bill Payment,4851,108419
1,Recharge,17149,299670


In [ ]:
z_prop2(17149,299670,4851,108419)

# p value < 0.05 so result is significant

z = 15.5963  p = 7.709e-55


In [38]:
# L1 · What are the churn rate and the monthly revenue of each region?
# STEPS
# 1. Group by region.
# 2. Count, sum revenue and average churn.
# 3. Sort by churn.
q("""SELECT region,
    COUNT(*) AS subs,
    SUM(is_churned) AS churn_rate
    FROM customer_features
    GROUP BY region
    ORDER BY churn_rate DESC;""")

,region,subs,churn_rate
0,North,7231,1716
1,West,4665,1299
2,East,5459,1196
3,South,4645,1015


In [ ]:
z_prop2(1716,7231,1015,4645)
# NOT significant 

z = 2.3755  p = 0.01752


In [42]:
# I4 · Which circles have the highest and the lowest churn rate?
# STEPS
# 1. Group by circle_name, carrying region.
# 2. Count and average churn.
# 3. Sort worst first and print all 22, because the spread is the finding.
q("""SELECT circle_name,region,
    COUNT(*) AS subs,
    SUM(is_churned) AS churn_rate
    FROM customer_features
    GROUP BY circle_name, region
    ORDER BY churn_rate DESC;""")

,circle_name,region,subs,churn_rate
0,Maharashtra,West,1428,470
1,Himachal Pradesh,North,1247,417
2,Andhra Pradesh,South,1412,408
3,Mumbai,West,1395,392
4,Assam,East,1301,301
5,Gujarat,West,1210,298
6,North East,East,1318,295
7,Haryana,North,1041,265
8,Uttar Pradesh (West),North,1081,248
9,Karnataka,South,1361,242


In [ ]:
z_prop2(470,1428,207,1082)
# NOT significant

z = 7.7046  p = 1.313e-14
